# Delhi Urban Safety Observatory — example analysis

Loads the manifest-documented yearly research releases (`data/releases/2016/` through `data/releases/2024/`), builds a single long-format panel across all nine years, and plots a basic citywide trend.

**Before reusing any number from this notebook**, read each year's `manifest.json` (source URLs, checksums, coverage, and null/comparability rules) — see the repo README's "Year-by-year research releases" section. The one rule this notebook takes seriously throughout: **a `NaN`/null value means the figure was not published for that district-year, never that the true count is zero.** Missing values are deliberately never filled with 0.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RELEASES = Path('..') / 'data' / 'releases'
YEARS = range(2016, 2025)

## 1. Load every year and build one long panel

Each `data/releases/<year>/district_crime.csv` has the same columns, so this is a straight concat — no reshaping needed. Each row is already one (year, district) pair, which is what "long format" / "tidy" means: safe to `groupby`, `pivot`, or feed straight into a regression without any melt/unmelt step first.

In [ ]:
frames = []
for year in YEARS:
    year_dir = RELEASES / str(year)
    frames.append(pd.read_csv(year_dir / 'district_crime.csv'))

panel = pd.concat(frames, ignore_index=True)
print(f'{len(panel)} district-year rows, {panel["year"].nunique()} years, {panel["district"].nunique()} districts')
panel.head()

## 2. Check coverage before trusting an aggregate

`coverage` and `null_reason` explain every gap. Skimming these first avoids the most common mistake with this dataset: silently summing across years where a district didn't exist as a separate reporting zone yet, or where a metric's source schema wasn't comparable, and mistaking the resulting drop for a real trend.

In [ ]:
panel['coverage'].value_counts()

In [ ]:
panel.loc[panel['coverage'] != 'reported', ['year', 'district', 'coverage', 'null_reason']]

## 3. Citywide trend — total IPC crime, reported rows only

Summing `total_ipc_bns` by year, restricted to `coverage == 'reported'` so an unreported district never gets treated as a zero. `total_ipc_previous_year_comparable` further flags which year-over-year steps are safe to read as real change versus a coverage artifact — plotted below as hollow markers.

In [ ]:
reported = panel[panel['coverage'] == 'reported']
citywide = reported.groupby('year')['total_ipc_bns'].sum(min_count=1)

comparable_years = (
    reported.groupby('year')['total_ipc_previous_year_comparable']
    .agg(lambda s: bool(s.all()))
)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(citywide.index, citywide.values, color='#e3a13b', linewidth=2, zorder=2)
comparable = citywide.index[comparable_years.reindex(citywide.index, fill_value=False)]
not_comparable = citywide.index.difference(comparable)
ax.scatter(comparable, citywide.loc[comparable], color='#e3a13b', zorder=3, label='year-over-year comparable')
ax.scatter(not_comparable, citywide.loc[not_comparable], facecolors='none', edgecolors='#b14a34', zorder=3, label='not directly comparable to prior year')
ax.set_title('Delhi citywide total IPC crime, reported districts only')
ax.set_xlabel('Year')
ax.set_ylabel('Total IPC crimes (sum across reported districts)')
ax.legend()
fig.tight_layout()
plt.show()

## 4. Merge in road-safety data for a second panel

`citywide_road_safety.csv` isn't broken down by district, so it's a natural companion to the crime panel rather than something to merge row-for-row with it: join on `year` alone for a combined citywide view.

In [ ]:
road_frames = []
for year in YEARS:
    year_dir = RELEASES / str(year)
    f = year_dir / 'citywide_road_safety.csv'
    if f.exists():
        road_frames.append(pd.read_csv(f))

road_safety = pd.concat(road_frames, ignore_index=True)
combined = citywide.rename('total_ipc_bns').reset_index().merge(road_safety, on='year', how='left')
combined

## 5. Where to go from here

- `data/releases/<year>/district_road_safety.csv` (2022-2024) and `crash_prone_zones.csv` (2023-2024) follow the same per-year layout and can be loaded/concatenated the same way as step 1.
- `data/releases/shared/manifest.json` catalogs non-year-specific layers (infrastructure, boundaries, liquor vends) with their own checksums and caveats — read its `warning` field before joining anything from there against a specific crime year.
- `data/releases/manifest.json` is the machine-readable index of everything above, if you want to automate this loading step instead of hardcoding `YEARS = range(2016, 2025)`.
- See the repo's `docs/DATA_CHANGELOG.md` for any corrections made to already-published figures since you last pulled this data.